# 02 | Victorian Council Data Quality Assessment

**Project:** Victorian Local Government Financial Sustainability & Performance Intelligence (2020–2025)

**Pipeline Stage:** Bronze Data Quality Assessment  
**Technology:** Databricks | PySpark | Delta Lake  
**Source Table:** `workspace.default.bronze_performance`

## Business Objective

Assess the completeness, consistency and reliability of Victorian council performance data before creating cleaned Silver datasets.

## Data Quality Checks

1. **Data structure:** Verify records, columns and data types.
2. **Missing values:** Identify missing information across all columns.
3. **Target availability:** Examine missing performance targets by financial year.
4. **Duplicate records:** Identify repeated records.
5. **Category consistency:** Check council group names and indicator classifications.
6. **Numeric validity:** Identify invalid performance results.

## Expected Outcome

A documented data quality assessment identifying issues requiring cleaning, standardisation or further investigation.

**Important:** This notebook analyses the Bronze data without modifying the original records.

In [0]:

# STEP 1: Import PySpark functions
from pyspark.sql import functions as F

# STEP 2: Read the Bronze Delta table
df = spark.table("workspace.default.bronze_performance")

# STEP 3: Check dataset size
print("Total rows:", df.count())
print("Total columns:", len(df.columns))

# STEP 4: Check column names and data types
df.printSchema()

# STEP 5: Preview the data
display(df.limit(10))


Total rows: 23068
Total columns: 10
root
 |-- Council: string (nullable = true)
 |-- Group: string (nullable = true)
 |-- ID: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Service_Provided: string (nullable = true)
 |-- Data_Applicable: string (nullable = true)
 |-- Result: string (nullable = true)
 |-- Variation: string (nullable = true)
 |-- FinancialYear: string (nullable = true)
 |-- Target: string (nullable = true)



Council,Group,ID,Description,Service_Provided,Data_Applicable,Result,Variation,FinancialYear,Target
Alpine Shire,Small Shire,AF2,Health inspections of aquatic facilities,YES,Applicable,1,All Council aquatic facilities received a formal health inspection during 2024/25.,2024-25,null
Alpine Shire,Small Shire,AF6,Utilisation of aquatic facilities,YES,Applicable,3.492840367,"Utilisation of Council's three aquatic facilities increased again in 2024/25, continuing a trend from the previous year, and surpassing Council's target result. Visitation at outdoor pools during the seasonal opening period (November - March) increased slightly compared to the previous year, in part due to a hotter summer season and a streamlined marketing program. The greatest driver of change was a 30% increase in visitation at Bright Sports Centre compared to the previous year. This was driven by an increase in member visitation during January - June 2025, in response to new equipment in the gymnasium section of the Centre.",2024-25,null
Alpine Shire,Small Shire,AF7,Cost of aquatic facilities,YES,Applicable,12.88104464,"While the cost per visit has decreased compared to the previous year, this is driven primarily by an increase in visitation which is larger than the increase in costs compared to the previous year. Investment in facilities (new gymnasium equipment) at the Bright Sports Centre appears to the be major driver of visitation increase.",2024-25,null
Alpine Shire,Small Shire,AM1,Time taken to action animal management requests,YES,Applicable,3.717687075,"Response times slightly increased compared to the previous year. Council responds to 66% of all Domestic Animals jobs in 1 day, 20% in 2-5 days, with the remaining 13% taking six or more days.",2024-25,null
Alpine Shire,Small Shire,AM2,Animals reclaimed,YES,Applicable,0.661971831,"Council continued to be successful in returning two-thirds of registered animals to their owners, a similar positive result to the previous year. The community utilises social media to assist in returning lost animals to their owners, meaning Council officers are generally called where these social media efforts are unsuccessful. Council undertakes an annual awareness campaign to ensure that pets are microchipped and registered.",2024-25,null
Alpine Shire,Small Shire,AM5,Animals rehomed,YES,Applicable,1,"Council increased its success in ensuring that all registerable cats and dogs that are collected find their way to a forever home. This is assisted by Council managing its own pound, and being able to rehome animals by advertising them on social media platforms.",2024-25,null
Alpine Shire,Small Shire,AM6,Cost of animal management service per population,YES,Applicable,10.39469884,"Cost of animal management service per population decreased slightly, mostly driven by a reduction in material costs which will vary from year to year.",2024-25,null
Alpine Shire,Small Shire,AM7,Animal management prosecutions,YES,Applicable,0,No animal management prosecutions were presented to court in 2024/25. This is consistent with Council's target for this measure.,2024-25,null
Alpine Shire,Small Shire,FS1,Time taken to action food complaints,YES,Applicable,1.75,"Council's Environmental Health team continued to focus on improved food safety awareness to promote compliance with Food Safety Standards, helping to keep the number of food complaints low.",2024-25,null
Alpine Shire,Small Shire,FS2,Food safety assessments,YES,Applicable,0.248618785,"Category 1 inspections were completed as a priority followed by a risk-based approach of Category 2, new food business applications, mandatory food sampling, and ensuring Council is meeting its statutory obligations under other Acts and Regulations.",2024-25,null


In [0]:

# STEP 1: Check missing values in each column
for column in df.columns:

    # Count null, blank and "null" text values
    missing = df.filter(
        F.col(f"`{column}`").isNull() |
        (F.trim(F.col(f"`{column}`")) == "") |
        (F.lower(F.trim(F.col(f"`{column}`"))) == "null")
    ).count()

    # Print the result
    print(column, ":", missing)


Council : 0
Group : 0
ID : 0
Description : 0
Service_Provided : 0
Data_Applicable : 0
Result : 142
Variation : 3922
FinancialYear : 0
Target : 21635


In [0]:

from pyspark.sql import functions as F

# Load Bronze data
df = spark.table("workspace.default.bronze_performance")

# Check missing values for each financial year
year_missing = df.groupBy("FinancialYear").agg(

    F.count("*").alias("Total_Rows"),

    F.sum(
        F.when(
            F.col("Result").isNull() |
            (F.lower(F.trim(F.col("Result"))) == "null") |
            (F.trim(F.col("Result")) == ""),
            1
        ).otherwise(0)
    ).alias("Missing_Result"),

    F.sum(
        F.when(
            F.col("Target").isNull() |
            (F.lower(F.trim(F.col("Target"))) == "null") |
            (F.trim(F.col("Target")) == ""),
            1
        ).otherwise(0)
    ).alias("Missing_Target")

).orderBy("FinancialYear")

display(year_missing)



# Check totals across all five years
totals = year_missing.agg(
    F.sum("Total_Rows").alias("rows"),
    F.sum("Missing_Result").alias("results"),
    F.sum("Missing_Target").alias("targets")
).first()

print("Total rows:", totals["rows"])
print("Missing Results:", totals["results"])
print("Missing Targets:", totals["targets"])

assert totals["rows"] == 23068
assert totals["results"] == 142
assert totals["targets"] == 21635

print("PASS: Year-level missingness is correct")


FinancialYear,Total_Rows,Missing_Result,Missing_Target
2020-21,4582,43,4582
2021-22,4582,99,4582
2022-23,4582,0,4582
2023-24,4661,0,4033
2024-25,4661,0,3856


Total rows: 23068
Missing Results: 142
Missing Targets: 21635
PASS: Year-level missingness is correct


In [0]:

# STEP 1: Count all records
total_rows = df.count()

# STEP 2: Count unique records
unique_rows = df.dropDuplicates().count()

# STEP 3: Calculate exact duplicate rows
duplicates = total_rows - unique_rows

# STEP 4: Display results
print("Total rows:", total_rows)
print("Unique rows:", unique_rows)
print("Exact duplicate rows:", duplicates)


Total rows: 23068
Unique rows: 23068
Exact duplicate rows: 0


In [0]:

# STEP 1: Group by Council, FinancialYear and Indicator ID
key_duplicates = (
    df.groupBy("Council", "FinancialYear", "ID")
    .count()
    .filter("count > 1")
)

# STEP 2: Count repeated combinations
print("Repeated council-year-indicator combinations:",
      key_duplicates.count())

# STEP 3: Display repeated combinations
display(key_duplicates.orderBy("Council", "FinancialYear", "ID"))


Repeated council-year-indicator combinations: 0


Council,FinancialYear,ID,count


In [0]:

# STEP 1: Count records in each council group
group_check = (
    df.groupBy("Group")
    .count()
    .orderBy("Group")
)

# STEP 2: Display council groups and record counts
display(group_check)

# STEP 3: Count different council groups
print("Total council groups:", df.select("Group").distinct().count())

# STEP 4: Count different councils
print("Total councils:", df.select("Council").distinct().count())


Group,count
INTERFACE,1044
Interface,1584
LARGE_SHIRE,2204
Large Shire,3344
METRO,2552
Metro,3872
REGIONAL,1160
Regional,590
Regional City,1170
SMALL_SHIRE,2204


Total council groups: 11
Total councils: 81


In [0]:

from pyspark.sql import functions as F

# Convert Result values into numbers
df_check = df.withColumn(
    "Result_Number",
    F.expr("try_cast(Result as double)")
)

# Find values that cannot be converted
invalid_results = df_check.filter(
    F.col("Result").isNotNull() &
    (F.trim(F.col("Result")) != "") &
    (F.lower(F.trim(F.col("Result"))) != "null") &
    F.col("Result_Number").isNull()
)

# Show the number of invalid values
print("Invalid numeric results:", invalid_results.count())

# Display any problematic records
display(invalid_results.limit(20))


Invalid numeric results: 0


Council,Group,ID,Description,Service_Provided,Data_Applicable,Result,Variation,FinancialYear,Target,Result_Number


In [0]:

# Check whether Target values can become numbers
df_target = df.withColumn(
    "Target_Number",
    F.expr("try_cast(Target as double)")
)

# Find non-empty Target values that failed conversion
invalid_targets = df_target.filter(
    F.col("Target").isNotNull() &
    (F.trim(F.col("Target")) != "") &
    (F.lower(F.trim(F.col("Target"))) != "null") &
    F.col("Target_Number").isNull()
)

# Show the results
print("Invalid numeric targets:", invalid_targets.count())

display(invalid_targets.limit(20))


Invalid numeric targets: 0


Council,Group,ID,Description,Service_Provided,Data_Applicable,Result,Variation,FinancialYear,Target,Target_Number



## Results and Outcomes

### Dataset Assessed
- Source: `workspace.default.bronze_performance`
- Records: 23,068
- Columns: 10
- Reporting period: 2020–21 to 2024–25

### Data Quality Results
- Missing Result values: 142
- Missing Target values: 21,635
- Missing Variation values: 3,922
- Exact duplicate records: 0
- Duplicate council-year-indicator combinations: 0
- Invalid non-missing Result values: 0
- Invalid non-missing Target values: 0
- Distinct council names: 81
- Distinct council group labels: 11

### Year-Level Findings
- Missing Results: 43 in 2020–21 and 99 in 2021–22.
- No missing Results from 2022–23 onward.
- All Targets are missing from 2020–21 to 2022–23.
- Target availability increases in 2023–24 and 2024–25.

### Reporting Implications
- Keep missing values as null rather than zero.
- Use valid Results for council performance trends.
- Calculate target achievement only where valid Targets exist.
- Standardise council names and groups before benchmarking.
- Explain historical data limitations in Power BI.

### Output
A documented data quality assessment.
The Bronze dataset remains unchanged.

### Next Steps
- Notebook 03: Compare source column names.
- Notebook 04: Validate council names and categories.
- Notebook 05: Create the cleaned Silver dataset.
